IPA completion attribution review. Run from the Data Manager workspace root. Read-only source comparison; writes diagnostic CSV only.

In [ ]:
import csv,json,collections
from pathlib import Path
base=Path('followup_dashboard/ipa_streamlit_app')
def read(name):
 with (base/'data/latest'/name).open() as f:return list(csv.DictReader(f))
records=read('participant_ipa_records.csv'); hist=read('ipa_appointment_history.csv'); audit=read('calendly_participant_match_audit.csv')
pending=[r for r in records if r['IPA Complete']=='TRUE' and r['Completion FTM']=='Pending verification']
if not pending:
 pending=[r for r in records if r['IPA Complete']=='TRUE' and 'No eligible host' in r['Completion Attribution']]
out=[]
for r in pending:
 h=[x for x in hist if x['Participant ID']==r['Participant ID']]
 a=[x for x in audit if x['Participant ID']==r['Participant ID'] and x['IPA Year'] in ['2025','2026']]
 sameband=[x for x in h if x['IPA Age Band']==r['IPA Age Band']]
 oldsame=[x for x in a if x['IPA Year']=='2025' and x['IPA Age Band']==r['IPA Age Band'] and x['Participant Cohort']==r['Participant Cohort']]
 day=[x for x in h if x['Event Start Date'][:10]==r['Completion Date'][:10] and x['Canceled']=='FALSE' and x['No Show']=='FALSE']
 if sameband and not any(x['Participant Cohort']==r['Participant Cohort'] for x in sameband): reason='2026 cohort mismatch'
 elif h:reason='2026 age-band mismatch'
 elif oldsame:reason='Only 2025 same-band record'
 elif a:reason='Only 2025 other-band record'
 else:reason='No participant-linked 2025–2026 assessment record'
 out.append({**r,'Reason':reason,'2026 Calendly rows':len(h),'2026 Calendly age bands':', '.join(sorted(set(x['IPA Age Band'] for x in h))),'2026 Calendly cohorts':', '.join(sorted(set(x['Participant Cohort'] for x in h))),'Same-day eligible host count':len(set(x['FTM'] for x in day)),'Same-day eligible hosts':', '.join(sorted(set(x['FTM'] for x in day))),'2025–2026 All linked rows':len(a)})
assert len(out)==20,len(out)
p=base/'outputs/ipa_completion_match_review_20261005';p.mkdir(parents=True,exist_ok=True)
with (p/'unresolved_ipa_completion_review.csv').open('w') as f:
 w=csv.DictWriter(f,fieldnames=list(out[0]));w.writeheader();w.writerows(out)
counts=collections.Counter(r['Reason'] for r in out)
print(json.dumps(dict(counts)));print('Same-day rows:',sum(r['Same-day eligible host count']>0 for r in out));print('Multiple same-day hosts:',sum(r['Same-day eligible host count']>1 for r in out))
